# core

> HTMX v4 support for FastHTML

In [ ]:
#| default_exp core

In [ ]:
#| export
import json

from fasthtml.common import *
from fasthtml.starlette import *
from fasthtml.core import *

from fastcore.basics import patch
from fastcore.utils import *
from fastcore.xml import *
from fastcore.meta import delegates




In [ ]:
#| export
@delegates(ft_hx)
def Partial(*args, **kwargs): return ft_hx("hx-partial")(*args, **kwargs)


In [ ]:
#| export
htmx4src   = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha6/dist/htmx.js")
ws4src = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha6/dist/ext/hx-ws.js")


In [ ]:
#| export
# When htmx4=True, configures htmx v4 with metaCharacter="-"
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    if htmx4: 
        # metaCharacter="-" makes htmx4 use dashes instead of colons (Python-friendly)
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
    # TODO: Check if fhjsscr works with htmx4
    return [charset, viewport] + hdrs

In [ ]:
#| export
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4=False` parameter to toggle htmx v4 headers
# - Passes htmx4 to def_hdrs() which handles the header selection
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr)
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super().__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

In [ ]:
#| export
def _get_tbl(dt, nm, schema):
    render = schema.pop('render', None)
    tbl = dt[nm]
    if tbl not in dt: tbl.create(**schema)
    else: tbl.create(**schema, transform=True)
    dc = tbl.dataclass()
    if render: dc.__ft__ = render
    return tbl,dc

def _app_factory(*args, **kwargs) -> FastHTML | FastHTMLWithLiveReload:
    "Creates a FastHTML or FastHTMLWithLiveReload app instance"
    if kwargs.pop('live', False): return FastHTMLWithLiveReload(*args, **kwargs)
    kwargs.pop('reload_attempts', None)
    kwargs.pop('reload_interval', None)
    return FastHTML(*args, **kwargs)

# Supports htmx4=True for htmx v4 compatibility
def fast_app(
        db_file:Optional[str]=None, # Database file name, if needed
        render:Optional[callable]=None, # Function used to render default database class
        hdrs:Optional[tuple]=None, # Additional FT elements to add to <HEAD>
        ftrs:Optional[tuple]=None, # Additional FT elements to add to end of <BODY>
        tbls:Optional[dict]=None, # Experimental mapping from DB table names to dict table definitions
        before:Optional[tuple]|Beforeware=None, # Functions to call prior to calling handler
        middleware:Optional[tuple]=None, # Standard Starlette middleware
        live:bool=False, # Enable live reloading
        debug:bool=False, # Passed to Starlette, indicating if debug tracebacks should be returned on errors
        title:str="FastHTML page", # Default page title
        routes:Optional[tuple]=None, # Passed to Starlette
        exception_handlers:Optional[dict]=None, # Passed to Starlette
        on_startup:Optional[callable]=None, # Passed to Starlette
        on_shutdown:Optional[callable]=None, # Passed to Starlette
        lifespan:Optional[callable]=None, # Passed to Starlette
        default_hdrs=True, # Include default FastHTML headers such as HTMX script?
        pico:Optional[bool]=None, # Include PicoCSS header?
        surreal:Optional[bool]=True, # Include surreal.js/scope headers?
        htmx:Optional[bool]=True, # Include HTMX header?
        htmx4:Optional[bool]=False, # Include HTMX4 header?
        exts:Optional[list|str]=None, # HTMX extension names to include
        canonical:bool=True, # Automatically include canonical link?
        secret_key:Optional[str]=None, # Signing key for sessions
        key_fname:str='.sesskey', # Session cookie signing key file name
        session_cookie:str='session_', # Session cookie name
        max_age:int=365*24*3600, # Session cookie expiry time
        sess_path:str='/', # Session cookie path
        same_site:str='lax', # Session cookie same site policy
        sess_https_only:bool=False, # Session cookie HTTPS only?
        sess_domain:Optional[str]=None, # Session cookie domain
        htmlkw:Optional[dict]=None, # Attrs to add to the HTML tag
        bodykw:Optional[dict]=None, # Attrs to add to the Body tag
        reload_attempts:Optional[int]=1, # Number of reload attempts when live reloading
        reload_interval:Optional[int]=1000, # Time between reload attempts in ms
        static_path:str=".",  # Where the static file route points to, defaults to root dir
        body_wrap:callable=noop_body, # FT wrapper for body contents
        nb_hdrs:bool=False, # If in notebook include headers inject headers in notebook DOM?
        **kwargs):
    "Create a FastHTML or FastHTMLWithLiveReload app."
    h = (picolink,) if pico or (pico is None and default_hdrs) else ()
    if hdrs: h += tuple(hdrs)

    app = _app_factory(hdrs=h, ftrs=ftrs, before=before, middleware=middleware, live=live, debug=debug, title=title, routes=routes, exception_handlers=exception_handlers,
                  on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan, default_hdrs=default_hdrs, secret_key=secret_key, canonical=canonical,
                  session_cookie=session_cookie, max_age=max_age, sess_path=sess_path, same_site=same_site, sess_https_only=sess_https_only,
                  sess_domain=sess_domain, key_fname=key_fname, exts=exts, surreal=surreal, htmx=htmx, htmx4=htmx4, htmlkw=htmlkw,
                  reload_attempts=reload_attempts, reload_interval=reload_interval, body_wrap=body_wrap, nb_hdrs=nb_hdrs, **(bodykw or {}))
    app.static_route_exts(static_path=static_path)
    if not db_file: return app,app.route

    db = database(db_file)
    if not tbls: tbls={}
    if kwargs:
        if isinstance(first(kwargs.values()), dict): tbls = kwargs
        else:
            kwargs['render'] = render
            tbls['items'] = kwargs
    dbtbls = [_get_tbl(db.t, k, v) for k,v in tbls.items()]
    if len(dbtbls)==1: dbtbls=dbtbls[0]
    return app,app.route,*dbtbls